# 💬 Чат с нашей нейросетью

Нажимай ▶ слева от ячеек **по очереди, сверху вниз**. Видеокарта не нужна.

Модель маленькая (14 млн параметров) и обучена с нуля за час — пишет по-русски, но часто отвечает невпопад. Это нормально 🙂

In [ ]:
#@title 1. Загрузить модель (≈30 секунд)
#@markdown Каждый запуск скачивает самую свежую версию проекта с GitHub.
import os, sys
REPO = '/content/ai'
BRANCH = 'ccr-d2dd0252-6ksj1l'
if os.path.exists(REPO):
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} reset -q --hard FETCH_HEAD
else:
    !git clone -q -b {BRANCH} https://github.com/r0561952-beep/ai.git {REPO}
!pip install -q tokenizers
%cd {REPO}
if REPO not in sys.path:
    sys.path.insert(0, REPO)
# забываем старые версии нашего кода, если ячейку запускали раньше
for m in ['model', 'tokenizer_utils', 'chat', 'web_chat']:
    sys.modules.pop(m, None)
from chat import ChatBot
bot = ChatBot()
print('Готово! Модель загружена, шаг обучения:', bot.ckpt['iter'])

In [ ]:
#@title 2. 💬 Открыть чат как в ChatGPT
#@markdown Появится ссылка вида **https://….gradio.live** — открой её, это чат во весь экран.
#@markdown Ссылка работает, пока эта ячейка запущена (до ~72 часов).
!pip install -q "gradio>=6,<7"
import sys
if 'bot' not in globals() or not hasattr(bot, 'stream'):
    print('⚠️ Сначала запусти ячейку 1 «Загрузить модель», потом эту.')
else:
    sys.modules.pop('web_chat', None)
    from web_chat import launch
    launch(share=True, bot=bot)

In [ ]:
#@title 3. Задать один вопрос (без интерфейса)
вопрос = "Как приготовить блины?" #@param {type:"string"}
температура = 0.7 #@param {type:"slider", min:0.1, max:1.5, step:0.1}
bot.temperature = температура
bot.reset()
print('Ты: ', вопрос)
print('Бот:', bot.answer(вопрос))

Температура: меньше — ответы спокойнее и однообразнее, больше — фантазийнее и безумнее.

In [ ]:
#@title 4. Диалог в текстовом виде (модель помнит последние реплики)
#@markdown Пиши в поле ниже. `/new` — начать заново, `/exit` — закончить.
bot.reset()
while True:
    q = input('Ты: ').strip()
    if q == '/exit':
        break
    if q == '/new':
        bot.reset(); print('(диалог забыт)'); continue
    if q:
        print('Бот:', bot.answer(q), '\n')